<a href="https://colab.research.google.com/github/sanju464/email_phising/blob/main/email_phising_using_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")

print(os.listdir(path))


Using Colab cache for faster access to the 'phishing-email-dataset' dataset.
['SpamAssasin.csv', 'Nazario.csv', 'Nigerian_Fraud.csv', 'CEAS_08.csv', 'Enron.csv', 'Ling.csv', '.nfs0000000017d0bbea0000000a', 'phishing_email.csv']


loding dataset

In [ ]:
df = pd.read_csv(os.path.join(path, "phishing_email.csv"))
df.head()


,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [ ]:
X = df['text_combined']
y = df['label']



split data into traning and test


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


TF-IDF Vectorization
TF-IDF reduces importance of common words
and increases importance of rare but meaningful words.
vectorisation convert text to numeric


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


actually training naive bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)


MultinomialNB()

evaluate

In [ ]:
from sklearn.metrics import classification_report

y_pred = nb.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.94      0.98      0.96      7919
           1       0.98      0.95      0.96      8579

    accuracy                           0.96     16498
   macro avg       0.96      0.96      0.96     16498
weighted avg       0.96      0.96      0.96     16498



test case

In [ ]:
def test_email(email):
    email_vector = vectorizer.transform([email])
    prediction = nb.predict(email_vector)[0]
    probability = nb.predict_proba(email_vector)[0]

    print("Email:", email)
    print("Prediction:", "Phishing" if prediction == 1 else "Legitimate")
    print("Confidence:", probability)
    print("-" * 60)


In [ ]:
test_email("Urgent! Your bank account has been suspended. Click here to verify your password immediately.")


Email: Urgent! Your bank account has been suspended. Click here to verify your password immediately.
Prediction: Phishing
Confidence: [0.01195832 0.98804168]
------------------------------------------------------------


In [ ]:
test_email("Your parcel delivery failed. Please confirm your details to reschedule shipment.")


Email: Your parcel delivery failed. Please confirm your details to reschedule shipment.
Prediction: Phishing
Confidence: [0.17018693 0.82981307]
------------------------------------------------------------


In [ ]:
test_email("Hi team, please find attached the meeting agenda for tomorrow's discussion.")


Email: Hi team, please find attached the meeting agenda for tomorrow's discussion.
Prediction: Legitimate
Confidence: [0.99347198 0.00652802]
------------------------------------------------------------


In [ ]:
test_email("Thank you for subscribing to our newsletter. Here are this week's updates.")


Email: Thank you for subscribing to our newsletter. Here are this week's updates.
Prediction: Legitimate
Confidence: [0.70398251 0.29601749]
------------------------------------------------------------


In [ ]:
test_email("We noticed unusual activity on your account. Verify your information to avoid suspension.")


Email: We noticed unusual activity on your account. Verify your information to avoid suspension.
Prediction: Phishing
Confidence: [0.17425618 0.82574382]
------------------------------------------------------------
